# 011-1. 실제 수요 비중 확대와 외부검증 예측순위 지도

GTX-A 공급구조 유지 250m 격자 126개를 역 권역별 카카오맵에 중첩한다. **역 권역별로 실제 결과와 쌍대 비교 순위모형의 중첩 LOSO 외부검증 지도를 나란히** 출력한다(운정중앙·킨텍스·대곡 3묶음).

- 실제 확대(1): 같은 역 권역에서 2025년 승차 비중이 2024년보다 큰 격자. 비확대(0): 같거나 작은 격자. **승차인원 자체 증가와 다르다.**
- 실제 지도 노란 테두리와 1·2·3: 해당 권역의 **실제 비중 증가폭(%p)** 상위 3개.
- 검증 지도 숫자: 해당 권역을 통째로 제외하고 학습한 모형의 **예측점수 순위**. 모든 격자를 지도에 나타내며, 상위 20%는 기존 Precision@20 검증과 대응하도록 참고 강조한다. 체크박스를 켜면 전체 순위 숫자가 보인다.
- 순위모형은 확대·비확대를 0/1로 판정하거나 확대확률을 내지 않는다. 따라서 상위 20%도 **확대 후보**이지 '확대 예측 확정 격자'가 아니다.
- 지도 및 OOF 점수는 파일로 저장하지 않는다.

In [1]:
# 프로젝트 루트(UROP) 자동 탐색 — 절대경로 없이 어느 컴퓨터에서도 동작
import sys
from pathlib import Path
UROP_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.py').is_file() and (p / 'code').is_dir())
for _d in [UROP_ROOT, *sorted(d for d in (UROP_ROOT / 'code').rglob('*') if d.is_dir() and d.name != '__pycache__')]:
    if str(_d) not in sys.path:
        sys.path.append(str(_d))

from pathlib import Path
import importlib.util
import os
import sys
import getpass
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

candidates = [*sorted(d for d in (UROP_ROOT / 'code').rglob('*') if d.is_dir()), 
    Path.cwd(),
    Path.cwd() / 'analysis',
    Path.cwd() / 'gtx_a_bus_analysis' / 'analysis',
    Path.cwd() / 'github_2026-UROP' / 'gtx_a_bus_analysis' / 'analysis',
]
ANALYSIS = next((p for p in candidates if (p / '12-8_A3_쌍대비교_순위모형.py').exists()), None)
if ANALYSIS is None:
    raise FileNotFoundError('GTX-A 분석 폴더를 찾을 수 없습니다. 노트북을 analysis 폴더에서 실행하세요.')
PROJECT = UROP_ROOT
ROOT = UROP_ROOT
if str(ANALYSIS) not in sys.path:
    sys.path.insert(0, str(ANALYSIS))

spec = importlib.util.spec_from_file_location('pairwise_rank_011_1', next((UROP_ROOT / 'code').rglob('12-8_A3_쌍대비교_순위모형.py')))
rank = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rank)
base = rank.load_base_module()
data = base.load_data().dropna(subset=['nearest_station', 'share_increase', *rank.FEATURES]).copy()
data['셀 ID'] = pd.to_numeric(data['셀 ID'], errors='raise').astype(int)
data['share_increase'] = data['share_increase'].astype(int)
assert len(data) == 126 and data['셀 ID'].nunique() == 126
print('분석 격자:', len(data), '| 실제 비중 확대:', int(data['share_increase'].sum()),
      '| 비확대:', int((data['share_increase'] == 0).sum()))
display(data.groupby('nearest_station')['share_increase'].agg(격자수='size', 실제확대수='sum'))

분석 격자: 126 | 실제 비중 확대: 66 | 비확대: 60


,격자수,실제확대수
nearest_station,,
대곡,37,18
운정중앙,63,34
킨텍스,26,14


## 1. 외부 검증역 예측점수 재계산

기존 12-8과 동일하게 각 검증역을 학습·전처리·C 선택에서 제외한다. **저장된 전체자료 최종모형은 이 지도에 사용하지 않는다.**

In [2]:
oof_parts = []
for test_station in ['운정중앙', '킨텍스', '대곡']:
    train = data[data['nearest_station'].astype(str) != test_station]
    test = data[data['nearest_station'].astype(str) == test_station].copy()
    best_c, _ = rank.choose_c_inner_loso(train)
    pair_x, pair_y = rank.make_pairs(train)
    fitted = rank.make_model(best_c).fit(pair_x, pair_y)
    test['ranking_score'] = rank.cell_scores(fitted, test)
    test['외부 선택 C'] = best_c
    test = test.sort_values(['ranking_score', '셀 ID'], ascending=[False, True])
    test['예측순위'] = np.arange(1, len(test) + 1)
    test['상위20% 후보'] = test['예측순위'] <= int(np.ceil(len(test) * .20))
    oof_parts.append(test[['셀 ID', 'nearest_station', 'ranking_score', '예측순위', '상위20% 후보', '외부 선택 C']])
oof = pd.concat(oof_parts, ignore_index=True)
assert len(oof) == 126 and oof['셀 ID'].nunique() == 126
print('126개 격자의 중첩 LOSO OOF 예측점수 재계산 완료')
display(oof.groupby('nearest_station').agg(
    검증격자=('셀 ID', 'size'), 상위20후보=('상위20% 후보', 'sum'),
    선택C=('외부 선택 C', 'first')
))

126개 격자의 중첩 LOSO OOF 예측점수 재계산 완료


,검증격자,상위20후보,선택C
nearest_station,,,
대곡,37,8,31.622777
운정중앙,63,13,0.100000
킨텍스,26,6,0.316228


## 2. 기존 011과 동일한 250m 격자 경계 연결

격자 ID는 원본 011과 **같은 공간자료·경계·정렬·중복 제거 절차**로 재구성한다. 실제 승차 비중과 OOF 점수는 셀 ID로 결합한다.

In [3]:
GRID_PATH = next((ROOT / 'data' / 'raw' / 'common' / 'grid' / '빈격자(250m)').glob('*.shp'))
ADMIN_PATH = ROOT / 'data' / 'raw' / 'common' / 'api_cache_gtx_a_routes' / 'HangJeongDong_ver20260701.geojson'
admin = gpd.read_file(ADMIN_PATH)
grid_crs = gpd.read_file(GRID_PATH, rows=1).crs
admin = admin.to_crs(grid_crs)
admin['sgg_code'] = admin['sgg'].astype(str).str.extract(r'([0-9]+)')[0].str.zfill(5)
city = admin[admin['sgg_code'].str.startswith(('4128', '4148'))]
boundary = city.geometry.union_all() if hasattr(city.geometry, 'union_all') else city.geometry.unary_union
grid = gpd.read_file(GRID_PATH, bbox=boundary.bounds)
grid = grid[grid.geometry.intersects(boundary)].copy()
grid['_key'] = grid.geometry.apply(lambda x: x.wkb if x is not None else None)
grid = grid.drop_duplicates('_key').drop(columns='_key').reset_index(drop=True)
grid['셀 ID'] = np.arange(1, len(grid) + 1)
plot = grid[['셀 ID', 'geometry']].merge(data, on='셀 ID', how='inner', validate='one_to_one')
plot = plot.merge(oof.drop(columns='nearest_station'), on='셀 ID', how='inner', validate='one_to_one')
assert len(plot) == 126 and plot['셀 ID'].nunique() == 126
assert plot[['share_pre', 'share_post', 'ranking_score']].notna().all().all()
print('지도 연결 완료:', len(plot), '개 격자')

station_coords = {
    '운정중앙': (37.716670, 126.728330),
    '킨텍스': (37.665000, 126.748060),
    '대곡': (37.631626, 126.811024),
}
top3_actual = (
    plot.assign(비중변화_p=pd.to_numeric(plot['share_post'] - plot['share_pre']) * 100)
    .query('share_increase == 1')
    .sort_values(['nearest_station', '비중변화_p', '셀 ID'], ascending=[True, False, True])
    .groupby('nearest_station', sort=False).head(3)
)
display(top3_actual[['nearest_station', '셀 ID', '비중변화_p', '예측순위']].round(3).reset_index(drop=True))

지도 연결 완료: 126 개 격자


,nearest_station,셀 ID,비중변화_p,예측순위
0,대곡,12461,5.162,8
1,대곡,12464,2.132,28
2,대곡,12725,1.014,23
3,운정중앙,10778,2.636,17
4,운정중앙,6421,2.596,37
5,운정중앙,15474,1.187,31
6,킨텍스,12765,3.989,11
7,킨텍스,9342,2.956,17
8,킨텍스,14062,0.626,4
